# 頭痛BERT（簡易版）: 遷移型BERT 1本だけを単発実行

元ノートブック `頭痛BERT_アブレーション比較.ipynb` の **5モデル比較（2×2＋baseline）と 5-fold CV を外した簡易版**。

**やること**：頭痛確定後の3分岐（急な痛み→しびれ→振る舞い）を **1本の共有BERT**（node-level FT・履歴なし）で辿る
**遷移型BERT を、患者単位 7:3 split で 1 回だけ**学習・評価する。

**軽量化**：`epochs=3→1`, `max_len=128→64`（速度優先）。複数 seed / CV はしない。

# 1. 環境セットアップ & データ準備

In [ ]:
# ============================================================
# 環境判定 & セットアップ (Colab / ローカル どちらでも動く)
# ============================================================
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==4.46.3', 'sentencepiece', 'fugashi',
                    'ipadic', 'unidic-lite', 'protobuf', 'tiktoken',
                    'pyyaml', 'japanize-matplotlib'], check=True)
    DATA_DIR = '/content/drive/MyDrive/NTCIR-19'
    CSV_PATH = f'{DATA_DIR}/headache_emergency_calls202605311132.csv'
    YAML_PATH = f'{DATA_DIR}/protocol.yaml'
else:
    BASE_DIR = 'C:/Users/hiyok/Desktop/Emergency_task'
    CSV_PATH = f'{BASE_DIR}/dataset/headache_emergency_calls202605311132.csv'
    YAML_PATH = f'{BASE_DIR}/transition_diagram/protocol.yaml'

print(f'CSV : {CSV_PATH}  (exists: {os.path.exists(CSV_PATH)})')
print(f'YAML: {YAML_PATH}  (exists: {os.path.exists(YAML_PATH)})')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
import pandas as pd
import re

df = pd.read_csv(CSV_PATH)
print(f'rows: {len(df)}')

def parse_conversation(conversation_text):
    dispatcher_turns = re.findall(r'Dispatcher:([^\n]*)', conversation_text)
    caller_turns = re.findall(r'Caller:([^\n]*)', conversation_text)
    qa_pairs = []
    for q, a in zip(dispatcher_turns, caller_turns):
        q, a = q.strip(), a.strip()
        if q and a:
            qa_pairs.append({'質問': q, '回答': a})
    return qa_pairs

df['qa_pairs'] = df['会話'].apply(parse_conversation)
df_e = df.explode('qa_pairs')
df_e['質問'] = df_e['qa_pairs'].apply(lambda x: x['質問'] if isinstance(x, dict) else None)
df_e['回答'] = df_e['qa_pairs'].apply(lambda x: x['回答'] if isinstance(x, dict) else None)
df_pairs = df_e.drop(columns=['会話', 'qa_pairs'])
df_pairs['ペア'] = df_pairs['質問'] + ' ' + df_pairs['回答']
df_pairs['ペア番号'] = df_pairs.groupby('id').cumcount()
df_pairs = df_pairs[['id', 'ペア番号', 'ペア', '痛み', 'しびれ', '振る舞い', 'トリアージ', '質問', '回答']].reset_index(drop=True)
print(f'pairs: {len(df_pairs)}')

## 1.2 yaml → branch_table（頭痛プロトコルの分岐構造）

In [ ]:
import yaml

with open(YAML_PATH, encoding='utf-8') as f:
    protocol = yaml.safe_load(f)
headache_proto = next(p for p in protocol['protocols'] if p['id'] == 'headache')

def is_branch_node(n):
    return 'choices' in n and not n.get('metadata_only', False)

branch_nodes = [n for n in headache_proto['nodes'] if is_branch_node(n)]
fallback_triage = headache_proto['fallback']['if_all_symptom_questions_negative']

def parse_choice(c):
    if c.get('triage'):
        return {'code': c['code'], 'text': c['text'], 'action': 'terminal', 'triage': c['triage']}
    if c.get('next'):
        return {'code': c['code'], 'text': c['text'], 'action': 'next', 'next_id': c['next']}
    return {'code': c['code'], 'text': c['text'], 'action': 'fallback', 'triage': fallback_triage}

branch_table = [{
    'id': n['id'],
    'question': n['question'],
    'suspected_condition': n.get('suspected_condition'),
    'choices': [parse_choice(c) for c in n['choices']]
} for n in branch_nodes]

branch_ids = {b['id'] for b in branch_table}
branch_to_label_col = {
    'headache_sudden_severe': '痛み',
    'headache_numbness_paralysis': 'しびれ',
    'headache_abnormal_behavior': '振る舞い',
}
LABEL_TO_CHOICE_CODE = {0: 'a', 1: 'b', 2: 'c'}
triage_decode = {0: 'R3', 1: 'R2', 2: 'Y2'}      # データのトリアージ値 → 文字列
TRIAGE_PRIORITY = {'R1': 6, 'R2': 5, 'R3': 4, 'Y1': 3, 'Y2': 2, 'G': 1}
print('branch_table:', [b['id'] for b in branch_table])
print('fallback:', fallback_triage)

## 1.3 Sentence-LUKE で文ベクトル化 → cos類似度 → 採用ペア選定

In [ ]:
from transformers import MLukeTokenizer, LukeModel
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm
tqdm.pandas()


class SentenceLukeJapanese:
    def __init__(self, name, device=None):
        self.tokenizer = MLukeTokenizer.from_pretrained(name)
        self.model = LukeModel.from_pretrained(name)
        self.model.eval()
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = torch.device(device)
        self.model.to(device)

    def _mean_pooling(self, out, mask):
        emb = out[0]
        m = mask.unsqueeze(-1).expand(emb.size()).float()
        return torch.sum(emb * m, 1) / torch.clamp(m.sum(1), min=1e-9)

    @torch.no_grad()
    def encode(self, sentences, batch_size=8):
        all_e = []
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i + batch_size]
            enc = self.tokenizer(batch, padding='longest', truncation=True, return_tensors='pt').to(self.device)
            out = self.model(**enc)
            all_e.extend(self._mean_pooling(out, enc['attention_mask']).to('cpu'))
        return torch.stack(all_e)


model_luke = SentenceLukeJapanese('sonoisa/sentence-luke-japanese-base-lite')

df_pairs['ペアのベクトル'] = df_pairs['ペア'].progress_apply(
    lambda x: model_luke.encode([x])[0].tolist() if pd.notna(x) else None
)

branch_emb = {b['id']: model_luke.encode([b['question']])[0].tolist() for b in branch_table}
conv_emb = np.array(df_pairs['ペアのベクトル'].tolist())
for b in branch_table:
    e = np.array(branch_emb[b['id']]).reshape(1, -1)
    df_pairs[f'cos_sim_{b["id"]}'] = cosine_similarity(conv_emb, e).flatten()
print('vectorize OK')

In [ ]:
# 採用ペア選定（cos類似度の gap 閾値方式・既存ロジック踏襲）
threshold = 0.02

def select_pairs_by_gap(g, col, th=0.02):
    sgi = g.sort_values(by=col, ascending=False)
    sgt = sgi.reset_index(drop=True)
    adopted = pd.Series(False, index=range(len(sgt)))
    if len(sgt) > 0:
        adopted.iloc[0] = True
    if len(sgt) <= 1:
        return adopted.set_axis(sgi.index).reindex(g.index)
    diffs = sgt[col].diff() * -1
    for i in range(1, len(diffs)):
        if diffs.iloc[i] > th:
            adopted.iloc[i:] = False
            break
        else:
            adopted.iloc[i] = True
    return adopted.set_axis(sgi.index).reindex(g.index)

for b in branch_table:
    ad = f'採用ペア_{b["id"]}'
    df_pairs[ad] = False
    for pid, group in df_pairs.groupby('id'):
        adoption = select_pairs_by_gap(group, f'cos_sim_{b["id"]}', threshold)
        df_pairs.loc[group.index, ad] = adoption

all_patient_ids = sorted(df_pairs['id'].unique())
print(f'採用ペア選定 OK / patients: {len(all_patient_ids)}')

# 2. 共通ヘルパー（state 構築・選択肢レンダリング）

In [ ]:
# 選択肢に「→ 行き先」を埋め込む（use_route=True のとき使用）
def render_choice_with_route(branch, choice):
    base = choice['text']
    if choice['action'] == 'terminal':
        sc = f'：{branch["suspected_condition"]}の疑い' if branch.get('suspected_condition') else ''
        return f'{base} → {choice["triage"]}{sc}'
    elif choice['action'] == 'next':
        next_b = next((x for x in branch_table if x['id'] == choice['next_id']), None)
        if next_b is not None:
            return f'{base} → 次の確認: 「{next_b["question"]}」'
        return f'{base} → 次の確認へ（{choice["next_id"]}）'
    return f'{base} → {choice["triage"]}（保留）'


# state（context）組み立て。フラグで構成要素を ON/OFF。
#   use_node_question : [現在の分岐] を入れるか（共有エンコーダがノードを識別するのに必須）
#   use_history       : [これまでの確認] を入れるか
def build_context(branch, adopted_pairs, history, use_node_question=True, use_history=True):
    parts = []
    if use_history and history:
        hist = '\n'.join([f'- {h["question"]} → {h["choice_text"]}' for h in history])
        parts.append(f'[これまでの確認]\n{hist}')
    if use_node_question:
        parts.append(f'[現在の分岐]\n{branch["question"]}')
    if adopted_pairs:
        parts.append(f'[患者の発話]\n{" ".join(adopted_pairs)}')
    if not parts:
        parts.append('(発話なし)')
    return '\n\n'.join(parts)


def adopted_pairs_of(pid, bid):
    ad = f'採用ペア_{bid}'
    return df_pairs[(df_pairs['id'] == pid) & (df_pairs[ad] == True)]['ペア'].tolist()


# under-triage 判定（予測が正解より緊急度が低い＝危険側の誤り）
def is_under_triage(true_t, pred_t):
    return TRIAGE_PRIORITY.get(pred_t, 0) < TRIAGE_PRIORITY.get(true_t, 0)


# ground-truth ラベルで歩いた gold path（分岐 → choice code の列）
def gold_path(pid, sub):
    path = []
    current = branch_table[0]['id']
    while True:
        b = next(x for x in branch_table if x['id'] == current)
        gt_code = LABEL_TO_CHOICE_CODE[int(sub[branch_to_label_col[b['id']]].iloc[0])]
        path.append((b['id'], gt_code))
        ch = next(c for c in b['choices'] if c['code'] == gt_code)
        if ch['action'] == 'next' and ch.get('next_id') in branch_ids:
            current = ch['next_id']
            continue
        break
    return path

# 3. 学習例の生成（ノード単位）

各患者・各ノードについて 1 例を作る。入力 context = [現在の分岐質問] + [採用ペア]（+履歴）、
label = そのノードの回答 index (0=はい/1=いいえ/2=不明)。全 (患者 × 3ノード) を例化（162×3=486 例・54/54/54 均衡）。

In [ ]:
def build_step_examples(use_node_question=True, use_history=False, use_route=True,
                       full_coverage=True):
    examples = []
    for pid, sub in df_pairs.groupby('id'):
        # 各ノードの gold choice
        gold = {}
        for b in branch_table:
            code = LABEL_TO_CHOICE_CODE[int(sub[branch_to_label_col[b['id']]].iloc[0])]
            gold[b['id']] = next(c for c in b['choices'] if c['code'] == code)

        if full_coverage:
            # 全ノードを順に例化（履歴は gold prefix）
            history = []
            for b in branch_table:
                ap = adopted_pairs_of(pid, b['id'])
                context = build_context(b, ap, history, use_node_question, use_history)
                ch_texts = [render_choice_with_route(b, c) if use_route else c['text']
                            for c in b['choices']]
                gt_idx = next(i for i, c in enumerate(b['choices']) if c['code'] == gold[b['id']]['code'])
                examples.append({'patient_id': pid, 'branch_id': b['id'],
                                 'context': context, 'choices': ch_texts, 'label': gt_idx})
                history.append({'branch_id': b['id'], 'question': b['question'],
                                'choice_text': gold[b['id']]['text']})
        else:
            # gold path を歩いて訪問ノードだけ
            history = []
            current = branch_table[0]['id']
            while True:
                b = next(x for x in branch_table if x['id'] == current)
                ap = adopted_pairs_of(pid, b['id'])
                context = build_context(b, ap, history, use_node_question, use_history)
                ch_texts = [render_choice_with_route(b, c) if use_route else c['text']
                            for c in b['choices']]
                gt_idx = next(i for i, c in enumerate(b['choices']) if c['code'] == gold[b['id']]['code'])
                examples.append({'patient_id': pid, 'branch_id': b['id'],
                                 'context': context, 'choices': ch_texts, 'label': gt_idx})
                gc = gold[b['id']]
                history.append({'branch_id': b['id'], 'question': b['question'],
                                'choice_text': gc['text']})
                if gc['action'] == 'next' and gc.get('next_id') in branch_ids:
                    current = gc['next_id']
                    continue
                break
    return pd.DataFrame(examples)


# 患者単位ランダム split 7:3
from sklearn.model_selection import train_test_split
TRAIN_IDS, TEST_IDS = train_test_split(all_patient_ids, test_size=0.3, random_state=42)
TEST_SET = set(TEST_IDS)
print(f'train患者={len(TRAIN_IDS)}, test患者={len(TEST_IDS)}')

_demo = build_step_examples()
print(f'例数(full_coverage)={len(_demo)} / branch分布={dict(_demo["branch_id"].value_counts())}')
print('label分布(全体):', dict(_demo["label"].value_counts().sort_index()))

# 4. 学習・推論の共通部品（軽量設定）

`epochs=1`, `max_len=64` に軽量化。

In [ ]:
from transformers import (BertJapaneseTokenizer, BertForSequenceClassification,
                          BertForMultipleChoice)
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
import torch.nn.functional as F

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BERT_NAME = 'cl-tohoku/bert-base-japanese-whole-word-masking'
print(f'device: {device}')

DEFAULT_CONFIG = {
    'name': 'unnamed',
    'head': 'seqcls',            # 'seqcls' / 'mc' / 'direct'
    'share_encoder': True,       # seqcls のみ: True=共有1本 / False=分岐ごと独立
    'use_node_question': True,   # context に [現在の分岐] を入れるか
    'use_history': False,        # 既定で履歴なし（node-level FT はマルコフ的）
    'use_route': True,           # mc のみ: 選択肢に行き先を埋め込むか
    'full_coverage': True,       # 全(患者×ノード)を学習例に（486例・均衡）
    'max_len': 64,               # ★簡易版: 128→64 に短縮
    'epochs': 1,                 # ★簡易版: 3→1 に短縮
    'lr': 2e-5,
    'batch_size': 4,
    'threshold': 0.0,            # 信頼度がこれ未満なら R3 で安全側終端（0=無効）
}


# log() が未定義でも動くようにフォールバック
if 'log' not in globals():
    def log(msg, also_print=True):
        if also_print:
            print(msg)


# ---- seqcls エンコード ----
def encode_seqcls(texts, labels, tokenizer, max_len):
    enc = tokenizer(list(texts), padding='max_length', truncation=True,
                    max_length=max_len, return_tensors='pt')
    return TensorDataset(enc['input_ids'], enc['attention_mask'], torch.tensor(list(labels)))


def train_seqcls(train_texts, train_labels, num_labels, cfg, tag=''):
    tok = BertJapaneseTokenizer.from_pretrained(BERT_NAME)
    model = BertForSequenceClassification.from_pretrained(
        BERT_NAME, num_labels=num_labels, attn_implementation='eager').to(device)
    opt = AdamW(model.parameters(), lr=cfg['lr'])
    dl = DataLoader(encode_seqcls(train_texts, train_labels, tok, cfg['max_len']),
                    batch_size=cfg['batch_size'], shuffle=True)
    for ep in range(cfg['epochs']):
        model.train()
        tot = 0
        for ids, am, lab in dl:
            ids, am, lab = ids.to(device), am.to(device), lab.to(device)
            opt.zero_grad()
            out = model(ids, attention_mask=am, labels=lab)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += out.loss.item()
        log(f'    [{tag}] seqcls epoch {ep + 1}/{cfg["epochs"]}  loss={tot / max(len(dl), 1):.4f}')
    return model, tok


@torch.no_grad()
def seqcls_predict(model, tok, text, n_labels, max_len):
    model.eval()
    enc = tok([text], padding='max_length', truncation=True,
              max_length=max_len, return_tensors='pt').to(device)
    probs = F.softmax(model(**enc).logits[0, :n_labels], dim=-1).cpu().numpy()
    idx = int(probs.argmax())
    return idx, float(probs[idx])


# ---- mc エンコード ----
NUM_CHOICES = max(len(b['choices']) for b in branch_table)

def encode_mc(df_split, tokenizer, max_len):
    ids_all, am_all, labs = [], [], []
    for _, row in df_split.iterrows():
        cs = list(row['choices'])
        while len(cs) < NUM_CHOICES:
            cs.append('')
        enc = tokenizer([row['context']] * NUM_CHOICES, cs,
                        padding='max_length', truncation=True,
                        max_length=max_len, return_tensors='pt')
        ids_all.append(enc['input_ids'])
        am_all.append(enc['attention_mask'])
        labs.append(row['label'])
    return TensorDataset(torch.stack(ids_all), torch.stack(am_all), torch.tensor(labs))


def train_mc(mc_train, cfg, tag=''):
    tok = BertJapaneseTokenizer.from_pretrained(BERT_NAME)
    model = BertForMultipleChoice.from_pretrained(BERT_NAME, attn_implementation='eager').to(device)
    opt = AdamW(model.parameters(), lr=cfg['lr'])
    dl = DataLoader(encode_mc(mc_train, tok, cfg['max_len']),
                    batch_size=cfg['batch_size'], shuffle=True)
    for ep in range(cfg['epochs']):
        model.train()
        tot = 0
        for ids, am, lab in dl:
            ids, am, lab = ids.to(device), am.to(device), lab.to(device)
            opt.zero_grad()
            out = model(input_ids=ids, attention_mask=am, labels=lab)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += out.loss.item()
        log(f'    [{tag}] mc epoch {ep + 1}/{cfg["epochs"]}  loss={tot / max(len(dl), 1):.4f}')
    return model, tok


@torch.no_grad()
def mc_predict(model, tok, context, rendered_choices, max_len):
    model.eval()
    n_valid = len(rendered_choices)
    cs = list(rendered_choices)
    while len(cs) < NUM_CHOICES:
        cs.append('')
    enc = tok([context] * NUM_CHOICES, cs, padding='max_length', truncation=True,
              max_length=max_len, return_tensors='pt')
    ids = enc['input_ids'].unsqueeze(0).to(device)
    am = enc['attention_mask'].unsqueeze(0).to(device)
    probs = F.softmax(model(input_ids=ids, attention_mask=am).logits[0, :n_valid], dim=-1).cpu().numpy()
    idx = int(probs.argmax())
    return idx, float(probs[idx])

# 5. 遷移図トラバーサル（greedy）と評価指標

In [ ]:
# choose(branch, context, rendered_choices) -> (choice_idx, confidence)
# を受け取り、遷移図を greedy に辿って最終 triage と経路を返す。
def traverse(pid, cfg, choose):
    history, pred_path = [], []
    current = branch_table[0]['id']
    while True:
        b = next(x for x in branch_table if x['id'] == current)
        ap = adopted_pairs_of(pid, b['id'])
        context = build_context(b, ap, history, cfg['use_node_question'], cfg['use_history'])
        rendered = [render_choice_with_route(b, c) if cfg['use_route'] else c['text']
                    for c in b['choices']]
        idx, conf = choose(b, context, rendered)
        if cfg['threshold'] > 0 and conf < cfg['threshold']:
            pred_path.append((b['id'], 'low_conf'))
            return 'R3', pred_path
        ch = b['choices'][idx]
        pred_path.append((b['id'], ch['code']))
        history.append({'branch_id': b['id'], 'question': b['question'],
                        'choice_text': ch['text']})
        if ch['action'] in ('terminal', 'fallback'):
            return ch['triage'], pred_path
        if ch.get('next_id') in branch_ids:
            current = ch['next_id']
            continue
        return fallback_triage, pred_path


def compute_metrics(res_df, node_records):
    # res_df: id / 真 / 予測 / in_test / 経路一致
    # node_records: list of {branch_id, correct, in_test}  (teacher-forced node判定)
    te = res_df[res_df['in_test']]
    overall = (te['真'] == te['予測']).mean()
    macro_f1 = f1_score(te['真'], te['予測'], average='macro', zero_division=0)
    under = te.apply(lambda r: is_under_triage(r['真'], r['予測']), axis=1).mean()
    path_match = te['経路一致'].mean()
    nd = pd.DataFrame(node_records)
    node_acc = nd[nd['in_test']]['correct'].mean() if len(nd) else float('nan')
    by_triage = (te.assign(c=te['真'] == te['予測']).groupby('真')['c'].mean().to_dict())
    by_node = (nd[nd['in_test']].groupby('branch_id')['correct'].mean().to_dict()
               if len(nd) else {})
    return {
        'overall_acc': overall, 'macro_f1': macro_f1, 'under_triage': under,
        'path_match': path_match, 'node_acc': node_acc,
        'by_triage': by_triage, 'by_node': by_node,
    }

# 6. `run_experiment(config)`（簡易版・チェックポイント無し）

In [ ]:
def run_experiment(config, train_ids=None, test_ids=None):
    cfg = {**DEFAULT_CONFIG, **config}
    train_ids = TRAIN_IDS if train_ids is None else train_ids
    test_ids = TEST_IDS if test_ids is None else test_ids
    test_set = set(test_ids)

    log(f'==== START [{cfg["name"]}]  head={cfg["head"]} share={cfg["share_encoder"]} '
        f'hist={cfg["use_history"]} route={cfg["use_route"]} epochs={cfg["epochs"]} ====')

    # ---------- M3: 直接BERT（遷移図なし） ----------
    if cfg['head'] == 'direct':
        direct = df_pairs.groupby('id').agg(
            text=('ペア', lambda x: ' '.join(x)), label=('トリアージ', 'first')).reset_index()
        tr = direct[direct['id'].isin(train_ids)]
        model, tok = train_seqcls(tr['text'], tr['label'].astype(int), 3, cfg, tag=cfg['name'])
        rows = []
        for _, r in direct.iterrows():
            idx, _ = seqcls_predict(model, tok, r['text'], 3, cfg['max_len'])
            rows.append({'id': r['id'], '真': triage_decode[int(r['label'])],
                         '予測': triage_decode[idx], 'in_test': r['id'] in test_set,
                         '経路一致': np.nan})
        res_df = pd.DataFrame(rows)
        m = compute_metrics(res_df, [])
        m.update({'name': cfg['name'], 'results': res_df, 'node_records': []})
        _report(m)
        return m

    # ---------- 学習例を生成 ----------
    ex = build_step_examples(cfg['use_node_question'], cfg['use_history'],
                             cfg['use_route'], cfg['full_coverage'])
    ex_tr = ex[ex['patient_id'].isin(train_ids)].reset_index(drop=True)

    # ---------- 学習 ----------
    if cfg['head'] == 'mc':
        model, tok = train_mc(ex_tr, cfg, tag=cfg['name'])
        def choose(b, context, rendered):
            return mc_predict(model, tok, context, rendered, cfg['max_len'])
        def choose_example(row):
            return mc_predict(model, tok, row['context'], row['choices'], cfg['max_len'])[0]

    elif cfg['share_encoder']:                      # 遷移型BERT（共有1本）
        model, tok = train_seqcls(ex_tr['context'], ex_tr['label'], 3, cfg, tag=cfg['name'])
        def choose(b, context, rendered):
            return seqcls_predict(model, tok, context, len(b['choices']), cfg['max_len'])
        def choose_example(row):
            b = next(x for x in branch_table if x['id'] == row['branch_id'])
            return seqcls_predict(model, tok, row['context'], len(b['choices']), cfg['max_len'])[0]

    else:                                           # 分岐ごと独立BERT
        per_branch = {}
        for bid, g in ex_tr.groupby('branch_id'):
            per_branch[bid] = train_seqcls(g['context'], g['label'], 3, cfg, tag=f'{cfg["name"]}/{bid}')
        def choose(b, context, rendered):
            mdl, tk = per_branch[b['id']]
            return seqcls_predict(mdl, tk, context, len(b['choices']), cfg['max_len'])
        def choose_example(row):
            mdl, tk = per_branch[row['branch_id']]
            return seqcls_predict(mdl, tk, row['context'], len(b['choices']), cfg['max_len'])[0]

    # ---------- ノード単位 acc（teacher-forced） ----------
    node_records = []
    for _, row in ex.iterrows():
        pred_idx = choose_example(row)
        node_records.append({'patient_id': row['patient_id'], 'branch_id': row['branch_id'],
                             'pred': int(pred_idx), 'gold': int(row['label']),
                             'correct': int(pred_idx == row['label']),
                             'in_test': row['patient_id'] in test_set})

    # ---------- 最終 triage（greedy トラバーサル） ----------
    rows = []
    for pid in all_patient_ids:
        sub = df_pairs[df_pairs['id'] == pid]
        pred_triage, pred_path = traverse(pid, cfg, choose)
        gpath = gold_path(pid, sub)
        rows.append({
            'id': pid,
            '真': triage_decode[int(sub['トリアージ'].iloc[0])],
            '予測': pred_triage,
            'in_test': pid in test_set,
            '経路一致': int(pred_path == gpath),
        })
    res_df = pd.DataFrame(rows)
    m = compute_metrics(res_df, node_records)
    m.update({'name': cfg['name'], 'results': res_df, 'node_records': node_records})
    _report(m)
    return m


def _report(m):
    print(f'  overall_acc = {m["overall_acc"]:.3f} | macro_f1 = {m["macro_f1"]:.3f} '
          f'| under_triage = {m["under_triage"]:.3f} '
          f'| node_acc = {m["node_acc"] if m["node_acc"]==m["node_acc"] else float("nan"):.3f} '
          f'| path_match = {m["path_match"]:.3f}')
    print(f'  by_triage = {{' + ', '.join(f"{k}:{v:.2f}" for k, v in m["by_triage"].items()) + '}}')

# 7. 簡易実行：遷移型BERT 1本だけ

頭痛確定後の3分岐を1本の共有BERT（node-level FT・履歴なし）で辿る本命モデルを、7:3 split で1回だけ学習・評価する。

In [ ]:
TRANS_CFG = {'name': '遷移型BERT (共有/履歴なし)',
             'head': 'seqcls', 'use_history': False, 'use_route': False}

result = run_experiment(TRANS_CFG)

# 8. 結果サマリ（最終トリアージ）

In [ ]:
# Jupyter外(headless)でも落ちないよう display をフォールバック
if 'display' not in globals():
    display = print

summary = pd.DataFrame([{
    'model': result['name'],
    'overall_acc': round(result['overall_acc'], 3),
    'macro_f1': round(result['macro_f1'], 3),
    'under_triage': round(result['under_triage'], 3),
    'node_acc': round(result['node_acc'], 3) if result['node_acc'] == result['node_acc'] else None,
    'path_match': round(result['path_match'], 3),
}])
display(summary)
print('by_triage(真ラベル別の最終triage acc):', {k: round(v, 3) for k, v in result['by_triage'].items()})
print('by_node(分岐別のノード単位acc)      :', {k: round(v, 3) for k, v in result['by_node'].items()})

import os
os.makedirs('output', exist_ok=True)
summary.to_csv('output/simple_summary.csv', index=False, encoding='utf-8-sig')
result['results'].to_csv('output/simple_per_patient.csv', index=False, encoding='utf-8-sig')
print('saved: output/simple_summary.csv, output/simple_per_patient.csv')

# 9. 分岐ごとに正しく選択肢を取れているか（node単位 / teacher-forced）

各分岐ノードを **gold文脈で独立に**判定したときの正解率と、どの選択肢へ誤ったかの混同行列（test集合）。
- acc が高い＝その分岐で正しい はい/いいえ/不明 を選べている
- 行 = 正解(gold) の選択肢、列 = モデル予測 の選択肢。対角が正解。

In [ ]:
nd = pd.DataFrame(result.get('node_records', []))
CHOICE_IDX = {0: 'a', 1: 'b', 2: 'c'}

if len(nd) == 0:
    print('node_records なし（直接BERT等）')
else:
    nd_te = nd[nd['in_test']].copy()
    print(f'=== 分岐ごとの node単位 acc（teacher-forced / test={nd_te["patient_id"].nunique() if "patient_id" in nd_te else "?"}）===')
    branch_rows = []
    for b in branch_table:
        sub = nd_te[nd_te['branch_id'] == b['id']]
        if len(sub) == 0:
            continue
        acc = sub['correct'].mean()
        labels = [f'{CHOICE_IDX[i]}:{c["text"]}' for i, c in enumerate(b['choices'])]
        branch_rows.append({'branch': b['id'], 'question': b['question'],
                            'node_acc': round(acc, 3), 'n': len(sub)})
        print(f'\n■ [{b["id"]}]  {b["question"]}')
        print(f'    node_acc = {acc:.3f}  (n={len(sub)})')
        cm = pd.crosstab(sub['gold'].map(lambda i: labels[i]),
                         sub['pred'].map(lambda i: labels[i]),
                         rownames=['gold↓'], colnames=['pred→'], dropna=False)
        print(cm.to_string())
    print('\n=== 分岐別サマリ ===')
    bdf = pd.DataFrame(branch_rows)
    display(bdf)
    bdf.to_csv('output/simple_per_branch.csv', index=False, encoding='utf-8-sig')
    print('saved: output/simple_per_branch.csv')